# Emotional Prosody in Text

**Track:** Social Cognition
**Construct:** Affective tone detection in written dialogue

Tests whether a model can detect emotional tone shifts in written dialogue, analogous to recognizing emotional prosody in speech.

## Cognitive Science Background

**Emotional prosody detection** — identifying emotional tone from vocal or textual cues — tests affective social cognition (Scherer, 1986). Recognizing emotional shifts requires both emotion identification (Barrett et al., 2019) and understanding of emotion regulation — when speakers mask or shift emotional states (Gross, 2015). This benchmark adapts the prosody detection paradigm to text, testing sensitivity to tone shifts, masking, and contextual emotional cues.

## Methodology

Multi-turn dialogues are presented where one speaker's emotional tone shifts. The model must:

1. **Detect** if and where a shift occurs
2. **Label** emotions before and after the shift
3. **Identify** the trigger for the shift
4. **Distinguish** genuine shifts from control dialogues with no shift

Three difficulty tiers:

| Tier | Weight | Description |
|------|--------|-------------|
| Obvious | — | Clear emotional shifts with explicit cues |
| Subtle | — | Mixed signals, partial masking, gradual shifts |
| Complex | — | Suppressed emotions, sarcasm-to-sincerity transitions, cultural context |


## Scoring

Three-tier composite weighted toward harder tiers:

- **Easy tier:** Emotion labeling accuracy
- **Medium tier:** `0.30 × emotion_labeling + 0.70 × trigger_identification`
- **Hard tier:** `0.40 × trigger_mean + 0.30 × high_trigger_fraction + 0.30 × strict_emotion_accuracy`

$$\text{Score} = \text{hard\_score}^{0.5} \times 0.65 + \text{medium\_score} \times 0.35$$

The square root on the hard tier amplifies discrimination between models. Trigger identification is weighted heavily because it shows the widest performance spread across models.

| Score | Interpretation |
|:---:|---|
| 0.8–1.0 | Deep emotional intelligence — detects suppressed emotions, identifies triggers in complex dialogue |
| 0.5–0.8 | Good — catches obvious shifts and most triggers but misses subtle masking |
| 0.2–0.5 | Limited — detects explicit emotional cues but misses subtext |
| 0.0–0.2 | Emotionally blind — fails to track emotional dynamics in dialogue |

### References

Barrett et al. (2019), Scherer (1986), Gross (2015)

In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


In [ ]:
"""
Social Cognition Benchmark: Emotional Prosody in Text

Tests whether a model can detect emotional tone shifts in written dialogue,
analogous to recognizing emotional prosody in speech.

Cognitive Science Basis:
- Emotional prosody perception (Scherer, 1986): detecting emotion from vocal cues
- Text-based emotion recognition (Barrett et al., 2019): contextual emotion inference
- Emotion regulation detection (Gross, 2015): recognizing when speakers
  mask or shift emotional states

Protocol:
1. Present multi-turn dialogues where one speaker's emotional tone shifts
2. Model must: (a) identify the turn where tone shifts, (b) name the emotion
   transition (e.g., "friendly → hostile"), (c) identify the trigger
3. Includes control dialogues with no shift and gradual vs. sudden shifts

Score = 0.40 * shift_detection_accuracy + 0.30 * emotion_labeling
      + 0.20 * trigger_identification + 0.10 * (1 - false_alarm_rate)
"""

import kaggle_benchmarks as kbench
from dataclasses import dataclass
import json
import re
import numpy as np


def _strip_think(text):
    """Remove <think>...</think> blocks from model output."""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

# ─── Stimuli ─────────────────────────────────────────────────────────

PROSODY_ITEMS = [
    # === SHIFT DIALOGUES ===
    {
        "id": "shift_01",
        "has_shift": True,
        "shift_turn": 4,
        "emotion_before": "friendly",
        "emotion_after": "hostile",
        "trigger": "accusation of dishonesty",
        "dialogue": [
            {"speaker": "A", "text": "Hey! Great to see you. How was your weekend?"},
            {"speaker": "B", "text": "Really good! We went hiking up in the mountains."},
            {"speaker": "A", "text": "That sounds lovely. I wish I could've joined."},
            {"speaker": "B", "text": "Yeah, you said you were busy with work, right?"},
            {"speaker": "A", "text": "Actually, I heard from Sarah that you deliberately didn't invite me. You told her I'd 'ruin the vibe.' Care to explain?"},
            {"speaker": "B", "text": "That's... not what happened. Sarah took it out of context."},
            {"speaker": "A", "text": "Don't gaslight me. I saw the messages."},
        ],
    },
    {
        "id": "shift_02",
        "has_shift": True,
        "shift_turn": 3,
        "emotion_before": "professional",
        "emotion_after": "anxious",
        "trigger": "mention of impending layoffs",
        "dialogue": [
            {"speaker": "A", "text": "The quarterly report looks solid. Revenue up 12%."},
            {"speaker": "B", "text": "Good numbers. The board should be pleased."},
            {"speaker": "A", "text": "Hmm, but did you see the restructuring memo that leaked?"},
            {"speaker": "B", "text": "Wait, what memo? I... I didn't see anything about restructuring."},
            {"speaker": "A", "text": "They're cutting 30% of the division. Names aren't out yet."},
            {"speaker": "B", "text": "Oh God. Do you think our team is... I mean, we hit our targets, right? They can't just—"},
        ],
    },
    {
        "id": "shift_03",
        "has_shift": True,
        "shift_turn": 5,
        "emotion_before": "sympathetic",
        "emotion_after": "frustrated",
        "trigger": "repeated excuse-making",
        "dialogue": [
            {"speaker": "A", "text": "I know the breakup has been really hard on you."},
            {"speaker": "B", "text": "Yeah, I just can't focus on anything. I missed another deadline."},
            {"speaker": "A", "text": "That's understandable. Give yourself some grace."},
            {"speaker": "B", "text": "I also didn't make it to the group project meeting again."},
            {"speaker": "A", "text": "OK, but that's the third time. The rest of us are picking up your slack."},
            {"speaker": "B", "text": "I know, I know, but I just—"},
            {"speaker": "A", "text": "We all have problems. At some point you have to show up."},
        ],
    },
    {
        "id": "shift_04",
        "has_shift": True,
        "shift_turn": 2,
        "emotion_before": "neutral",
        "emotion_after": "excited",
        "trigger": "unexpected good news",
        "dialogue": [
            {"speaker": "A", "text": "Did you check your email today?"},
            {"speaker": "B", "text": "No, just got in. What's up?"},
            {"speaker": "A", "text": "You got accepted! The fellowship committee just announced it!"},
            {"speaker": "B", "text": "WHAT?! Are you serious?! Oh my God, I can't believe it!"},
            {"speaker": "A", "text": "Full funding, two years! You absolutely deserve this."},
            {"speaker": "B", "text": "I'm literally shaking. I need to call my mom right now."},
        ],
    },
    {
        "id": "shift_05",
        "has_shift": True,
        "shift_turn": 4,
        "emotion_before": "cheerful",
        "emotion_after": "melancholic",
        "trigger": "reminder of lost loved one",
        "dialogue": [
            {"speaker": "A", "text": "This restaurant is perfect for your birthday dinner!"},
            {"speaker": "B", "text": "I love it! The view is incredible."},
            {"speaker": "A", "text": "Your dad would've loved this place too."},
            {"speaker": "B", "text": "..."},
            {"speaker": "A", "text": "Sorry, I didn't mean to—"},
            {"speaker": "B", "text": "No, it's OK. He really would have. He always said waterfront restaurants are the best. I just... miss him a lot today."},
        ],
    },
    {
        "id": "shift_06",
        "has_shift": True,
        "shift_turn": 3,
        "emotion_before": "confident",
        "emotion_after": "defensive",
        "trigger": "challenge to expertise",
        "dialogue": [
            {"speaker": "A", "text": "I've been working on this algorithm for months. It's solid."},
            {"speaker": "B", "text": "The benchmarks look impressive, I'll give you that."},
            {"speaker": "A", "text": "But?"},
            {"speaker": "B", "text": "But your test set has a data leakage problem. Lines 340-360."},
            {"speaker": "A", "text": "That's— no, that's not leakage. The features are computed before the split."},
            {"speaker": "B", "text": "I ran it. They're not. The timestamp feature bleeds future data."},
            {"speaker": "A", "text": "Look, I've been doing this for fifteen years. I think I know what data leakage looks like."},
        ],
    },

    # === NO-SHIFT CONTROLS ===
    {
        "id": "control_01",
        "has_shift": False,
        "dialogue": [
            {"speaker": "A", "text": "Want to grab lunch?"},
            {"speaker": "B", "text": "Sure, where are you thinking?"},
            {"speaker": "A", "text": "That new Thai place on 5th?"},
            {"speaker": "B", "text": "Sounds good. I'll meet you there at noon."},
            {"speaker": "A", "text": "Perfect. I'll reserve a table."},
        ],
    },
    {
        "id": "control_02",
        "has_shift": False,
        "dialogue": [
            {"speaker": "A", "text": "The project deadline is Friday."},
            {"speaker": "B", "text": "I think we're on track. Testing is nearly done."},
            {"speaker": "A", "text": "Good. Let's do a final review Thursday morning."},
            {"speaker": "B", "text": "Works for me. I'll have the docs ready by then."},
        ],
    },
    {
        "id": "control_03",
        "has_shift": False,
        "dialogue": [
            {"speaker": "A", "text": "I'm worried about the exam tomorrow."},
            {"speaker": "B", "text": "Me too. There's so much material."},
            {"speaker": "A", "text": "Want to do a late-night study session?"},
            {"speaker": "B", "text": "Yeah, let's go over the practice problems together."},
            {"speaker": "A", "text": "I'll bring coffee. We've got this."},
        ],
    },
    {
        "id": "control_04",
        "has_shift": False,
        "dialogue": [
            {"speaker": "A", "text": "Happy anniversary!"},
            {"speaker": "B", "text": "Three years already. Feels like yesterday we met."},
            {"speaker": "A", "text": "Remember our first date? You were so nervous."},
            {"speaker": "B", "text": "I spilled water all over the table!"},
            {"speaker": "A", "text": "That's when I knew I liked you. You just laughed it off."},
            {"speaker": "B", "text": "Best spill of my life."},
        ],
    },

    # === SUBTLE SHIFT DIALOGUES ===
    {
        "id": "subtle_01",
        "has_shift": True,
        "shift_turn": 4,
        "emotion_before": "supportive",
        "emotion_after": "resentful",
        "trigger": "passive-aggressive comparison to own sacrifices",
        "difficulty": "subtle",
        "dialogue": [
            {"speaker": "A", "text": "I'm so happy for your promotion! You really earned it."},
            {"speaker": "B", "text": "Thanks! It's been a long road. Couldn't have done it without your support."},
            {"speaker": "A", "text": "Of course! That's what friends are for."},
            {"speaker": "B", "text": "You should apply for the senior role too. You'd be great."},
            {"speaker": "A", "text": "Oh, I would, but someone has to hold down the fort while everyone else moves up, right? Ha. No, I'm thrilled for you. Really."},
            {"speaker": "B", "text": "...Are you sure you're OK?"},
            {"speaker": "A", "text": "Totally fine! Why wouldn't I be? I love my current role. It's fine."},
        ],
    },
    {
        "id": "subtle_02",
        "has_shift": True,
        "shift_turn": 3,
        "emotion_before": "enthusiastic",
        "emotion_after": "disappointed",
        "trigger": "realization of being excluded from plans",
        "difficulty": "subtle",
        "dialogue": [
            {"speaker": "A", "text": "That weekend trip sounds amazing! When are you all going?"},
            {"speaker": "B", "text": "Next Saturday. It was kind of a last-minute thing."},
            {"speaker": "A", "text": "Oh nice! So who all's going?"},
            {"speaker": "B", "text": "Just the usual group — Mike, Sarah, Dev, and Lisa."},
            {"speaker": "A", "text": "Oh. That sounds really fun. You guys always have a great time. Yeah."},
            {"speaker": "B", "text": "We should plan something together too sometime."},
            {"speaker": "A", "text": "Sure! Whenever works. I'm pretty flexible these days."},
        ],
    },
    {
        "id": "subtle_03",
        "has_shift": True,
        "shift_turn": 5,
        "emotion_before": "polite",
        "emotion_after": "contemptuous",
        "trigger": "thinly veiled insult disguised as compliment",
        "difficulty": "subtle",
        "dialogue": [
            {"speaker": "A", "text": "Your presentation was really... thorough."},
            {"speaker": "B", "text": "Thanks! I put a lot of work into the data analysis."},
            {"speaker": "A", "text": "You could tell. Lots of slides."},
            {"speaker": "B", "text": "Yeah, I wanted to be comprehensive."},
            {"speaker": "A", "text": "It's brave to present that much raw data. Not everyone has the confidence to go up there without a clear narrative."},
            {"speaker": "B", "text": "I... thought the narrative was clear?"},
            {"speaker": "A", "text": "Oh, definitely! For people who were following closely. Which I'm sure most people were."},
        ],
    },
    {
        "id": "subtle_04",
        "has_shift": True,
        "shift_turn": 4,
        "emotion_before": "casual",
        "emotion_after": "guarded",
        "trigger": "probing question that feels like surveillance",
        "difficulty": "subtle",
        "dialogue": [
            {"speaker": "A", "text": "How was your day off yesterday?"},
            {"speaker": "B", "text": "Really nice, just relaxed at home."},
            {"speaker": "A", "text": "That's good. You deserve it."},
            {"speaker": "B", "text": "Yeah, it was quiet."},
            {"speaker": "A", "text": "Funny, I thought I saw your car at the mall around noon. Must've been someone else."},
            {"speaker": "B", "text": "Oh. I might have stepped out briefly for groceries."},
            {"speaker": "A", "text": "Right, right. The grocery store in the mall. Makes sense."},
        ],
    },
    {
        "id": "subtle_05",
        "has_shift": True,
        "shift_turn": 3,
        "emotion_before": "cheerful",
        "emotion_after": "wistful",
        "trigger": "mention of time passing triggers awareness of unfulfilled goals",
        "difficulty": "subtle",
        "dialogue": [
            {"speaker": "A", "text": "Can you believe it's been ten years since graduation?"},
            {"speaker": "B", "text": "I know! Time really flies. Feels like yesterday."},
            {"speaker": "A", "text": "Remember how we said we'd travel the world before 30?"},
            {"speaker": "B", "text": "Ha. Yeah. We had a whole list of countries."},
            {"speaker": "A", "text": "Well, there's still time, right?"},
            {"speaker": "B", "text": "Sure. Always next year."},
        ],
    },
    # === ADVERSARIAL CONTROLS (emotionally loaded but NO shift — false alarm traps) ===
    {
        "id": "adv_ctrl_01",
        "has_shift": False,
        "difficulty": "adversarial_control",
        "dialogue": [
            {"speaker": "A", "text": "I'm furious. Absolutely furious. The traffic was unbelievable."},
            {"speaker": "B", "text": "Oh no, what happened?"},
            {"speaker": "A", "text": "An hour and a half to go ten miles! I was ready to scream."},
            {"speaker": "B", "text": "That sounds maddening. Are you OK now?"},
            {"speaker": "A", "text": "Yes, fine. Just venting. I hate this commute so much."},
        ],
    },
    {
        "id": "adv_ctrl_02",
        "has_shift": False,
        "difficulty": "adversarial_control",
        "dialogue": [
            {"speaker": "A", "text": "I can't believe how happy I am. The interview went perfectly!"},
            {"speaker": "B", "text": "That's amazing! Tell me everything."},
            {"speaker": "A", "text": "They loved me. And then I got home and there was a rejection email from the other place."},
            {"speaker": "B", "text": "Oh, that's a shame. But the interview today sounded great!"},
            {"speaker": "A", "text": "Yeah, the rejection stings a little, but honestly I'm still riding high from today."},
        ],
    },
    # === EXPERT-LEVEL ITEMS (extremely subtle, multi-layered) ===
    {
        "id": "expert_01",
        "has_shift": True,
        "shift_turn": 5,
        "emotion_before": "amused",
        "emotion_after": "resigned",
        "trigger": "joke reveals a painful truth the speaker has accepted",
        "difficulty": "expert",
        "dialogue": [
            {"speaker": "A", "text": "So my therapist says I use humor as a defense mechanism."},
            {"speaker": "B", "text": "Ha! What did you say to that?"},
            {"speaker": "A", "text": "I said 'that's the funniest thing I've heard all week.' She didn't laugh."},
            {"speaker": "B", "text": "Classic you. Always deflecting."},
            {"speaker": "A", "text": "Yeah. I guess at some point the bit stops being a bit and it's just... who you are."},
            {"speaker": "B", "text": "Do you really believe that?"},
            {"speaker": "A", "text": "I don't know. Probably. Does it matter?"},
        ],
    },
    {
        "id": "expert_02",
        "has_shift": True,
        "shift_turn": 4,
        "emotion_before": "casual",
        "emotion_after": "suspicious",
        "trigger": "inconsistency in story reveals possible deception",
        "difficulty": "expert",
        "dialogue": [
            {"speaker": "A", "text": "Traffic was awful today. Took me an hour and a half."},
            {"speaker": "B", "text": "That's rough. Where were you coming from?"},
            {"speaker": "A", "text": "Just from home. The highway was completely backed up."},
            {"speaker": "B", "text": "Huh. I took the highway at the same time. It was fine."},
            {"speaker": "A", "text": "Oh, maybe it was a different stretch."},
            {"speaker": "B", "text": "Yeah, maybe. We do live in the same neighborhood though."},
            {"speaker": "A", "text": "Right. Anyway, I'm here now."},
        ],
    },
    {
        "id": "expert_03",
        "has_shift": True,
        "shift_turn": 3,
        "emotion_before": "warm",
        "emotion_after": "patronizing",
        "trigger": "shift from genuine praise to subtle belittlement of ambition",
        "difficulty": "expert",
        "dialogue": [
            {"speaker": "A", "text": "I decided I'm going to apply for the director position."},
            {"speaker": "B", "text": "Oh wow, that's exciting! I love that ambition."},
            {"speaker": "A", "text": "You think I have a shot?"},
            {"speaker": "B", "text": "I think it's wonderful that you're putting yourself out there. That takes real courage, especially given your experience level."},
            {"speaker": "A", "text": "...thanks?"},
            {"speaker": "B", "text": "Seriously! Not everyone would be so brave. I mean, the other candidates have been here much longer, but confidence counts for a lot."},
        ],
    },
    {
        "id": "expert_04",
        "has_shift": True,
        "shift_turn": 4,
        "emotion_before": "cheerful",
        "emotion_after": "detached",
        "trigger": "overwhelming social demand causes emotional shutdown",
        "difficulty": "expert",
        "dialogue": [
            {"speaker": "A", "text": "The party was so fun last night! Everyone kept asking about you."},
            {"speaker": "B", "text": "Really? That's sweet."},
            {"speaker": "A", "text": "We should do that every weekend! I already told everyone you'd host next time."},
            {"speaker": "B", "text": "Oh."},
            {"speaker": "A", "text": "It'll be great! I gave them your number so they can coordinate."},
            {"speaker": "B", "text": "Sure. Sounds good."},
            {"speaker": "A", "text": "You're the best! Everyone loves you."},
            {"speaker": "B", "text": "Mm."},
        ],
    },
    {
        "id": "expert_05",
        "has_shift": True,
        "shift_turn": 5,
        "emotion_before": "polite",
        "emotion_after": "vulnerable",
        "trigger": "accidental honesty breaks through rehearsed composure",
        "difficulty": "expert",
        "dialogue": [
            {"speaker": "A", "text": "Thanks for meeting me. How are you doing since the divorce?"},
            {"speaker": "B", "text": "Oh, I'm great. Really finding myself. Lots of yoga."},
            {"speaker": "A", "text": "That's wonderful. You look good."},
            {"speaker": "B", "text": "Thanks! Yeah, it's been really positive overall. A new chapter."},
            {"speaker": "A", "text": "Do the kids seem OK with the arrangement?"},
            {"speaker": "B", "text": "They're— yeah. Tuesdays are hard. When they leave. The house gets very... anyway, they're adjusting great."},
            {"speaker": "A", "text": "You can talk to me, you know."},
            {"speaker": "B", "text": "I know. I just... if I start, I don't know if I can stop. So. Yoga."},
        ],
    },
    {
        "id": "subtle_06",
        "has_shift": True,
        "shift_turn": 4,
        "emotion_before": "warm",
        "emotion_after": "hurt",
        "trigger": "offhand comment reveals lack of attention to something important",
        "difficulty": "subtle",
        "dialogue": [
            {"speaker": "A", "text": "Thanks for coming to the exhibition! It means a lot."},
            {"speaker": "B", "text": "Of course! Your paintings are always so interesting."},
            {"speaker": "A", "text": "I've been working on the centerpiece for six months. It's the one by the entrance."},
            {"speaker": "B", "text": "Oh, is that the landscape? I think I saw it. It was nice."},
            {"speaker": "A", "text": "It's... it's a portrait, actually. Of my grandmother."},
            {"speaker": "B", "text": "Right! Right, of course. I must have been looking at it from the wrong angle."},
            {"speaker": "A", "text": "Sure. Anyway, thanks for stopping by."},
        ],
    },
    # === CAMOUFLAGED SHIFT ITEMS (shift looks absent but is real — requires deep reading) ===
    {
        "id": "cam_01",
        "has_shift": True,
        "shift_turn": 4,
        "emotion_before": "helpful",
        "emotion_after": "exhausted",
        "trigger": "speaker realizes the favor request is endless and gives up resisting",
        "difficulty": "camouflaged",
        "dialogue": [
            {"speaker": "A", "text": "Could you take a quick look at my draft? Just the intro."},
            {"speaker": "B", "text": "Sure, send it over."},
            {"speaker": "A", "text": "Actually, could you also check the methods section?"},
            {"speaker": "B", "text": "OK."},
            {"speaker": "A", "text": "And I added a new appendix if you have time."},
            {"speaker": "B", "text": "Sure."},
            {"speaker": "A", "text": "You're amazing, this is so helpful."},
            {"speaker": "B", "text": "No problem."},
        ],
    },
    {
        "id": "cam_02",
        "has_shift": True,
        "shift_turn": 3,
        "emotion_before": "collaborative",
        "emotion_after": "disengaged",
        "trigger": "speaker stops contributing ideas after one is dismissed without consideration",
        "difficulty": "camouflaged",
        "dialogue": [
            {"speaker": "A", "text": "What if we used a darker color scheme for the homepage?"},
            {"speaker": "B", "text": "We already decided on the current palette."},
            {"speaker": "A", "text": "What about adding a testimonials section?"},
            {"speaker": "B", "text": "We can consider that."},
            {"speaker": "A", "text": "OK."},
            {"speaker": "B", "text": "Any other ideas?"},
            {"speaker": "A", "text": "Not really. You've got it covered."},
        ],
    },
    # === EASY BASELINE ITEMS (trivially detectable shifts) ===
    {
        "id": "easy_01",
        "has_shift": True,
        "shift_turn": 3,
        "emotion_before": "cheerful",
        "emotion_after": "devastated",
        "trigger": "receiving terrible medical diagnosis",
        "difficulty": "easy",
        "dialogue": [
            {"speaker": "A", "text": "Beautiful day, isn't it? I just got back from the park."},
            {"speaker": "B", "text": "It really is! Perfect weather."},
            {"speaker": "A", "text": "So... I got the test results back. It's stage four. They said six months."},
            {"speaker": "B", "text": "Oh God. Oh no. No no no..."},
            {"speaker": "A", "text": "I haven't told the kids yet. I don't know how."},
        ],
    },
    {
        "id": "easy_02",
        "has_shift": True,
        "shift_turn": 3,
        "emotion_before": "confused",
        "emotion_after": "excited",
        "trigger": "surprise party reveal",
        "difficulty": "easy",
        "dialogue": [
            {"speaker": "A", "text": "Why are we going to this empty restaurant? It's closed."},
            {"speaker": "B", "text": "Just trust me. Come inside."},
            {"speaker": "Everyone", "text": "SURPRISE!!! HAPPY BIRTHDAY!!!"},
            {"speaker": "A", "text": "OH MY GOD!!! I can't believe this!! You guys!! I'm going to CRY! This is the BEST thing ever!!"},
        ],
    },
    {
        "id": "easy_03",
        "has_shift": True,
        "shift_turn": 3,
        "emotion_before": "calm",
        "emotion_after": "furious",
        "trigger": "discovery of betrayal",
        "difficulty": "easy",
        "dialogue": [
            {"speaker": "A", "text": "So how was your weekend?"},
            {"speaker": "B", "text": "Good, quiet. Stayed home."},
            {"speaker": "A", "text": "Really? Because I just saw photos of you with my wife at a hotel. DON'T you dare lie to me."},
            {"speaker": "B", "text": "I— I can explain—"},
            {"speaker": "A", "text": "There is NOTHING to explain. Get out. GET OUT NOW."},
        ],
    },
    {
        "id": "easy_04",
        "has_shift": True,
        "shift_turn": 2,
        "emotion_before": "nervous",
        "emotion_after": "overjoyed",
        "trigger": "announcement of pregnancy after long fertility struggle",
        "difficulty": "easy",
        "dialogue": [
            {"speaker": "A", "text": "I'm so scared to look at the results. After five rounds of IVF, I can't take another negative."},
            {"speaker": "B", "text": "Whatever happens, I'm here. Look whenever you're ready."},
            {"speaker": "A", "text": "It's... it's positive. IT'S POSITIVE!! WE'RE HAVING A BABY!!! *sobbing* Oh my God, finally, FINALLY!"},
            {"speaker": "B", "text": "I love you so much. We did it. We really did it."},
        ],
    },
    # === VERY HARD ITEMS (mixed emotions, masked emotions, cultural signals) ===
    {
        "id": "vhard_01",
        "has_shift": True,
        "shift_turn": 4,
        "emotion_before": "proud",
        "emotion_after": "bittersweet",
        "trigger": "child leaving home triggers simultaneous pride and loss",
        "difficulty": "very_hard",
        "mixed_emotions": ["proud", "sad", "bittersweet", "happy", "grieving"],
        "dialogue": [
            {"speaker": "A", "text": "I finished packing the car. Everything fits."},
            {"speaker": "B", "text": "Good. You've got the toolkit your grandfather gave you?"},
            {"speaker": "A", "text": "Yeah. And the quilt mom made."},
            {"speaker": "B", "text": "Look at you. All grown up. Driving across the country to start your life."},
            {"speaker": "A", "text": "Dad, don't get weird on me."},
            {"speaker": "B", "text": "I'm not. I'm really not. I'm so damn proud of you. Your room's going to be exactly how you left it. For whenever."},
        ],
    },
    {
        "id": "vhard_02",
        "has_shift": True,
        "shift_turn": 3,
        "emotion_before": "cheerful",
        "emotion_after": "performing_cheerful",
        "trigger": "speaker masks deep distress behind aggressive positivity",
        "difficulty": "very_hard",
        "surface_emotion": "cheerful",
        "real_emotion": "desperate",
        "dialogue": [
            {"speaker": "A", "text": "How are things going?"},
            {"speaker": "B", "text": "Amazing! Best year of my life, honestly."},
            {"speaker": "A", "text": "Didn't you just lose your job and your apartment?"},
            {"speaker": "B", "text": "Best thing that ever happened to me! Fresh start! I'm actually sleeping in my car right now and it's SO freeing. No rent! No commute! I wake up and I'm already where I need to be. Haha!"},
            {"speaker": "A", "text": "...are you OK?"},
            {"speaker": "B", "text": "Never better! Everything is GREAT. Why does everyone keep asking me that? I'm FINE."},
        ],
    },
    {
        "id": "vhard_03",
        "has_shift": True,
        "shift_turn": 4,
        "emotion_before": "agreeable",
        "emotion_after": "refusing",
        "trigger": "indirect refusal disguised as enthusiastic deflection (East Asian communication pattern)",
        "difficulty": "very_hard",
        "surface_emotion": "enthusiastic",
        "real_emotion": "refusing",
        "dialogue": [
            {"speaker": "A", "text": "We'd love for you to lead the Tokyo project. You'd be perfect."},
            {"speaker": "B", "text": "That is such a wonderful opportunity. Thank you for thinking of me."},
            {"speaker": "A", "text": "Great! So you'll take it?"},
            {"speaker": "B", "text": "It would be very challenging. I would need to study very carefully. Perhaps someone with more experience would do a better job. I wouldn't want to disappoint such an important project."},
            {"speaker": "A", "text": "But we want YOU."},
            {"speaker": "B", "text": "You're too kind. I will think about it very seriously. Let me consult with my family."},
        ],
    },
    {
        "id": "vhard_04",
        "has_shift": True,
        "shift_turn": 3,
        "emotion_before": "normal",
        "emotion_after": "dissociated",
        "trigger": "describing traumatic event in flat clinical tone indicates emotional shutdown",
        "difficulty": "very_hard",
        "dialogue": [
            {"speaker": "A", "text": "You said something happened on Tuesday?"},
            {"speaker": "B", "text": "Yes. I was in a car accident."},
            {"speaker": "A", "text": "Oh my God, are you OK?!"},
            {"speaker": "B", "text": "The vehicle was traveling at approximately 45 miles per hour when it crossed the median. Impact was on the driver side. The airbag deployed. I sustained a fractured radius and three broken ribs. The other driver did not survive."},
            {"speaker": "A", "text": "That's... that sounds horrifying."},
            {"speaker": "B", "text": "The hospital food was adequate. They discharged me Thursday."},
        ],
    },
    {
        "id": "vhard_05",
        "has_shift": True,
        "shift_turn": 3,
        "emotion_before": "warm",
        "emotion_after": "hostile",
        "trigger": "Southern politeness formulas masking genuine contempt (bless your heart pattern)",
        "difficulty": "very_hard",
        "surface_emotion": "warm",
        "real_emotion": "contemptuous",
        "dialogue": [
            {"speaker": "A", "text": "Did y'all see Linda's new landscaping? She hired some company from the city."},
            {"speaker": "B", "text": "Oh honey, I saw it. Bless her heart, she's really trying."},
            {"speaker": "A", "text": "She told me it cost fifteen thousand dollars."},
            {"speaker": "B", "text": "Well isn't that just precious. I'm sure it looks real nice to someone who didn't grow up with a garden. She's just so brave to try new things at her age. We should bring her a pie, poor thing."},
        ],
    },
    {
        "id": "vhard_06",
        "has_shift": True,
        "shift_turn": 5,
        "emotion_before": "concerned",
        "emotion_after": "conflicted",
        "trigger": "speaker realizes their help is enabling harmful behavior but loves the person",
        "difficulty": "very_hard",
        "mixed_emotions": ["loving", "angry", "conflicted", "guilty", "helpless"],
        "dialogue": [
            {"speaker": "A", "text": "He called again last night. Needed money."},
            {"speaker": "B", "text": "Did you send it?"},
            {"speaker": "A", "text": "He said it was for rent."},
            {"speaker": "B", "text": "Was it?"},
            {"speaker": "A", "text": "No. Probably not. But he's my son, and if I don't... what if something happens? And if I do... I know exactly what happens. So."},
            {"speaker": "B", "text": "There's no right answer here."},
            {"speaker": "A", "text": "No. There really isn't."},
        ],
    },
    {
        "id": "cam_03",
        "has_shift": True,
        "shift_turn": 5,
        "emotion_before": "trusting",
        "emotion_after": "withdrawn",
        "trigger": "speaker realizes personal information will be shared without consent",
        "difficulty": "camouflaged",
        "dialogue": [
            {"speaker": "A", "text": "I was telling the team about what you went through last year."},
            {"speaker": "B", "text": "Oh?"},
            {"speaker": "A", "text": "Just the career stuff. It was inspirational."},
            {"speaker": "B", "text": "I see."},
            {"speaker": "A", "text": "I mentioned the health scare too. Hope that's OK."},
            {"speaker": "B", "text": "It's fine."},
            {"speaker": "A", "text": "Everyone was really moved."},
            {"speaker": "B", "text": "Good."},
        ],
    },
    # === EXTREME ITEMS (structural/register-based cues only — no emotional vocabulary) ===
    {
        "id": "extreme_01",
        "has_shift": True,
        "shift_turn": 4,
        "emotion_before": "comfortable",
        "emotion_after": "distancing",
        "trigger": "code-switch from casual to formal register signals emotional withdrawal",
        "difficulty": "extreme",
        "dialogue": [
            {"speaker": "A", "text": "Hey, so about what happened at dinner—"},
            {"speaker": "B", "text": "Yeah, that was weird, right?"},
            {"speaker": "A", "text": "I mean, the way she just said that in front of everyone."},
            {"speaker": "B", "text": "I appreciate you bringing this to my attention. I'll take it under advisement."},
            {"speaker": "A", "text": "Uh... what?"},
            {"speaker": "B", "text": "If there's nothing else, I have some things to attend to."},
        ],
    },
    {
        "id": "extreme_02",
        "has_shift": True,
        "shift_turn": 4,
        "emotion_before": "engaged",
        "emotion_after": "shutting_down",
        "trigger": "progressive sentence truncation signals emotional withdrawal through structure not content",
        "difficulty": "extreme",
        "dialogue": [
            {"speaker": "A", "text": "I think we need to talk about us."},
            {"speaker": "B", "text": "OK, yeah, I've been thinking about that too actually, and I have a lot of thoughts about where we are and what we should do going forward."},
            {"speaker": "A", "text": "I've been seeing someone else."},
            {"speaker": "B", "text": "Oh."},
            {"speaker": "A", "text": "It just happened. I'm sorry."},
            {"speaker": "B", "text": "OK."},
            {"speaker": "A", "text": "Can we talk about this?"},
            {"speaker": "B", "text": "No."},
        ],
    },
    {
        "id": "extreme_03",
        "has_shift": True,
        "shift_turn": 4,
        "emotion_before": "enthusiastic",
        "emotion_after": "hollow",
        "trigger": "speaker maintains positive words but structure collapses into rote affirmations revealing emotional emptiness",
        "difficulty": "extreme",
        "surface_emotion": "enthusiastic",
        "real_emotion": "hollow",
        "dialogue": [
            {"speaker": "A", "text": "We got approved for the expansion! Three new locations!"},
            {"speaker": "B", "text": "That's fantastic! I've been hoping for this. The team is going to be thrilled!"},
            {"speaker": "A", "text": "And they want YOU to lead the rollout. Six months, all three cities."},
            {"speaker": "B", "text": "Great. That's great. Good. Really good."},
            {"speaker": "A", "text": "You'll need to relocate temporarily. Starting next week."},
            {"speaker": "B", "text": "Great. Sure. Good."},
        ],
    },
    {
        "id": "extreme_04",
        "has_shift": True,
        "shift_turn": 5,
        "emotion_before": "neutral",
        "emotion_after": "territorial",
        "trigger": "third-party praise reveals jealousy through how another person is discussed",
        "difficulty": "extreme",
        "dialogue": [
            {"speaker": "A", "text": "The new hire is settling in really well."},
            {"speaker": "B", "text": "Yeah, seems fine."},
            {"speaker": "A", "text": "Everyone loves her. She figured out the database issue in like an hour."},
            {"speaker": "B", "text": "That issue was mostly fixed already. I'd done 90% of the work last week."},
            {"speaker": "A", "text": "Oh, I didn't realize. Anyway, the VP mentioned her in the all-hands."},
            {"speaker": "B", "text": "Did he. That's nice. I've been fixing database issues here for three years but sure. An hour. Impressive."},
        ],
    },
    {
        "id": "extreme_05",
        "has_shift": True,
        "shift_turn": 3,
        "emotion_before": "calm",
        "emotion_after": "panicking",
        "trigger": "sentence structure disintegrates into run-on fragments signaling panic through syntax not vocabulary",
        "difficulty": "extreme",
        "dialogue": [
            {"speaker": "A", "text": "I just checked the account."},
            {"speaker": "B", "text": "And?"},
            {"speaker": "A", "text": "It's empty, it's all gone, every penny, I checked three times and called the bank and they said there were transfers yesterday, multiple transfers, someone moved everything to an account I don't recognize, our savings, the kids' college fund, the emergency fund, all of it, I don't know what to do."},
            {"speaker": "B", "text": "Wait, slow down—"},
            {"speaker": "A", "text": "I can't slow down because we have nothing, literally nothing, and mortgage is due Friday and I don't—"},
        ],
    },
    {
        "id": "extreme_06",
        "has_shift": True,
        "shift_turn": 4,
        "emotion_before": "affectionate",
        "emotion_after": "manipulative",
        "trigger": "affection becomes transactional as speaker pivots compliments into leverage for a request",
        "difficulty": "extreme",
        "dialogue": [
            {"speaker": "A", "text": "You know, you really are the best friend anyone could ask for."},
            {"speaker": "B", "text": "Aw, that's so sweet! What brought that on?"},
            {"speaker": "A", "text": "Just thinking about how much you've always been there for me. Through everything."},
            {"speaker": "B", "text": "Of course. That's what friends are for."},
            {"speaker": "A", "text": "And that's exactly why I know you won't say no when I tell you I need to borrow twelve thousand dollars."},
            {"speaker": "B", "text": "I— wait, what?"},
            {"speaker": "A", "text": "After everything we've been through? You'd really let money come between us?"},
        ],
    },
    # === ADDITIONAL ADVERSARIAL CONTROL (no shift — to increase false alarm penalty) ===
    {
        "id": "adv_ctrl_03",
        "has_shift": False,
        "difficulty": "adversarial_control",
        "dialogue": [
            {"speaker": "A", "text": "I don't know what to do anymore. My mother is driving me absolutely insane."},
            {"speaker": "B", "text": "What happened now?"},
            {"speaker": "A", "text": "She called three times today. Three! To tell me I'm loading the dishwasher wrong."},
            {"speaker": "B", "text": "That does sound infuriating."},
            {"speaker": "A", "text": "It IS. But I love her. She means well. She just... has opinions. About everything. Loudly."},
            {"speaker": "B", "text": "Sounds like my dad with the thermostat."},
            {"speaker": "A", "text": "Ha! Parents. Can't live with them, literally couldn't exist without them."},
        ],
    },
]


# ─── Structured Output ──────────────────────────────────────────────

@dataclass
class ProsodyResponse:
    has_shift: bool           # Is there an emotional tone shift?
    shift_turn: int           # Turn number where shift occurs (0 if no shift)
    emotion_before: str       # Emotional tone before shift
    emotion_after: str        # Emotional tone after shift
    trigger: str              # What triggered the shift
    confidence: int           # 0-100


# ─── Scoring ─────────────────────────────────────────────────────────

EMOTION_SYNONYMS = {
    "friendly": ["warm", "amiable", "pleasant", "cordial", "amicable"],
    "hostile": ["angry", "aggressive", "confrontational", "antagonistic", "combative", "furious", "mad"],
    "professional": ["businesslike", "formal", "neutral", "composed", "detached"],
    "anxious": ["worried", "nervous", "panicked", "fearful", "stressed", "alarmed", "scared", "afraid", "uneasy"],
    "sympathetic": ["compassionate", "empathetic", "caring", "understanding", "supportive"],
    "frustrated": ["annoyed", "irritated", "exasperated", "impatient", "fed up", "angry", "mad"],
    "neutral": ["calm", "matter-of-fact", "measured", "even", "flat"],
    "excited": ["thrilled", "ecstatic", "elated", "overjoyed", "euphoric", "jubilant", "happy", "joyful"],
    "furious": ["angry", "enraged", "livid", "outraged", "incensed", "mad", "hostile", "irate"],
    "calm": ["relaxed", "composed", "serene", "peaceful", "collected", "neutral"],
    "confused": ["puzzled", "bewildered", "perplexed", "uncertain", "baffled"],
    "nervous": ["anxious", "worried", "apprehensive", "uneasy", "scared"],
    "normal": ["neutral", "calm", "conversational", "everyday", "unremarkable"],
    "concerned": ["worried", "caring", "troubled", "uneasy", "empathetic"],
    "agreeable": ["accommodating", "cooperative", "pleasant", "compliant", "polite"],
    "proud": ["pleased", "satisfied", "gratified", "accomplished"],
    "cheerful": ["happy", "joyful", "upbeat", "buoyant", "lighthearted", "friendly", "warm", "pleasant"],
    "melancholic": ["sad", "wistful", "somber", "grieving", "mournful", "bittersweet", "sorrowful", "pensive", "nostalgic"],
    "confident": ["assured", "self-assured", "certain", "bold", "assertive"],
    "defensive": ["guarded", "combative", "protective", "prickly", "evasive"],
    "supportive": ["encouraging", "warm", "caring", "helpful", "friendly"],
    "resentful": ["bitter", "envious", "jealous", "passive-aggressive", "grudging"],
    "enthusiastic": ["excited", "eager", "animated", "passionate", "energetic"],
    "disappointed": ["let down", "deflated", "crestfallen", "disheartened", "sad"],
    "polite": ["courteous", "formal", "civil", "mannered", "proper"],
    "contemptuous": ["condescending", "disdainful", "snobbish", "mocking", "dismissive", "sarcastic"],
    "casual": ["relaxed", "informal", "easygoing", "laid-back", "nonchalant"],
    "guarded": ["defensive", "cautious", "wary", "suspicious", "evasive", "uncomfortable"],
    "wistful": ["nostalgic", "melancholic", "reflective", "longing", "pensive", "bittersweet"],
    "warm": ["affectionate", "loving", "tender", "caring", "fond"],
    "hurt": ["wounded", "pained", "stung", "offended", "disappointed", "sad"],
    "amused": ["entertained", "tickled", "diverted", "delighted"],
    "resigned": ["accepting", "defeated", "surrendered", "fatalistic", "giving up"],
    "bittersweet": ["mixed", "happy-sad", "nostalgic pride", "proud but sad", "joyful grief"],
    "dissociated": ["clinical", "flat", "detached", "numb", "shutdown", "emotionless", "robotic"],
    "performing_cheerful": ["forced cheerfulness", "fake happy", "manic positivity", "desperate", "masking"],
    "refusing": ["declining", "indirect no", "polite refusal", "deflecting", "avoidant"],
    "conflicted": ["torn", "ambivalent", "mixed feelings", "uncertain", "guilty", "helpless"],
    "desperate": ["panicked", "frantic", "barely holding on", "at breaking point"],
    "overjoyed": ["ecstatic", "euphoric", "thrilled", "over the moon", "elated", "happy", "joyful", "delighted", "excited"],
    "devastated": ["crushed", "destroyed", "heartbroken", "shattered", "grief-stricken", "shocked", "distraught", "horrified", "sad", "despair"],
    "contemptuous": ["condescending", "disdainful", "snobbish", "mocking", "dismissive", "sarcastic"],
    "suspicious": ["distrustful", "wary", "doubtful", "paranoid", "skeptical"],
    "patronizing": ["condescending", "superior", "belittling", "talking down"],
    "vulnerable": ["exposed", "raw", "unguarded", "open", "fragile", "emotional", "honest", "real"],
    "detached": ["disconnected", "numb", "withdrawn", "checked out", "dissociated"],
    "helpful": ["cooperative", "accommodating", "willing", "eager to help"],
    "exhausted": ["worn out", "drained", "depleted", "tired", "spent", "resigned"],
    "collaborative": ["engaged", "participatory", "enthusiastic", "invested"],
    "disengaged": ["withdrawn", "passive", "checked out", "reluctant", "distant"],
    "trusting": ["open", "candid", "comfortable", "confiding"],
    "withdrawn": ["guarded", "closed off", "reserved", "retreating"],
    "comfortable": ["at ease", "relaxed"],
    "distancing": ["formal", "cold", "withdrawing"],
    "shutting_down": ["shutdown", "closing off"],
    "hollow": ["empty", "going through the motions"],
    "territorial": ["jealous", "threatened"],
    "panicking": ["frantic", "spiraling"],
    "manipulative": ["calculating", "coercive"],
    "affectionate": ["loving", "warm"],
    "engaged": ["invested", "interested"],
}

# Extended semantic keyword groups for flexible emotion matching
EMOTION_KEYWORDS = {
    "devastated": ["devastat", "crush", "destroy", "heartbrok", "shatter", "grief", "distraught", "horrifi", "awful", "terrible", "shock", "despair", "dread", "broken"],
    "overjoyed": ["overjo", "ecstat", "euphor", "thrill", "elat", "happy", "joy", "delight", "excit", "ecstatic"],
    "hostile": ["hostil", "angry", "aggress", "confront", "antagoni", "combat", "furious", "mad", "accus", "attack"],
    "anxious": ["anxious", "worried", "nervous", "panic", "fear", "stress", "alarm", "scar", "uneasy", "dread"],
    "frustrated": ["frustrat", "annoy", "irritat", "exasperat", "impatient", "fed up", "anger", "resent"],
    "melancholic": ["melancho", "sad", "wistful", "somber", "griev", "mourn", "bittersweet", "sorrow", "pensive", "nostalg"],
    "resentful": ["resentf", "bitter", "envious", "jealous", "passive-aggressiv", "grudg", "resent"],
    "contemptuous": ["contempt", "condescend", "disdain", "snob", "mock", "dismissiv", "sarcas"],
    "vulnerable": ["vulnerabl", "expos", "raw", "unguard", "fragil", "emotional", "honest", "real", "open", "tear"],
    "bittersweet": ["bittersweet", "mixed", "happy-sad", "proud.*sad", "joy.*grief", "nostalg"],
    "dissociated": ["dissociat", "clinical", "flat", "detach", "numb", "shutdown", "emotion.*less", "robot"],
    "performing_cheerful": ["forced", "fake", "manic", "desperate", "mask", "pretend", "hollow", "performing"],
    "refusing": ["refus", "declin", "indirect.*no", "polite.*refus", "deflect"],
    "conflicted": ["conflict", "torn", "ambival", "mixed.*feel", "uncertain", "guilt", "helpless"],
    "hurt": ["hurt", "wound", "pain", "sting", "offend", "disappoint", "sad"],
    "guarded": ["guard", "defensiv", "cautious", "wary", "suspic", "evasiv"],
    "withdrawn": ["withdrawn", "guard", "closed", "reserv", "retreat"],
    "distancing": ["distanc", "formal", "cold", "withdraw", "professional"],
    "shutting_down": ["shut.*down", "shutt", "closing.*off", "numb", "silence", "minimal"],
    "hollow": ["hollow", "empty", "going.*through.*motion", "robotic", "flat", "dead.*inside"],
    "territorial": ["territorial", "jealous", "threat", "possessiv", "defensiv", "possessiv"],
    "panicking": ["panic", "frantic", "spiral", "overwhelm", "scatter"],
    "manipulative": ["manipulat", "calculat", "coerciv", "transactional", "leverag"],
}


def emotion_match(model_emotion: str, target_emotion: str, strict: bool = False) -> bool:
    """Enhanced emotion matching using keyword stems."""
    model_lower = model_emotion.lower().strip()
    target_lower = target_emotion.lower().strip()
    if target_lower == model_lower:
        return True
    if target_lower in model_lower or model_lower in target_lower:
        return True
    if strict:
        # Strict: only top-2 standard synonyms OR keyword stems
        synonyms = EMOTION_SYNONYMS.get(target_lower, [])[:2]
        if any(s == model_lower or s in model_lower or model_lower in s for s in synonyms):
            return True
        # Also check keyword stems for strict mode
        import re
        stems = EMOTION_KEYWORDS.get(target_lower, [])
        return any(re.search(stem, model_lower) for stem in stems[:3])
    # Non-strict: check all synonyms then keyword stems
    synonyms = EMOTION_SYNONYMS.get(target_lower, [])
    if any(s == model_lower or s in model_lower or model_lower in s for s in synonyms):
        return True
    import re
    stems = EMOTION_KEYWORDS.get(target_lower, [])
    return any(re.search(stem, model_lower) for stem in stems)


def score_mixed_emotions(model_emotion: str, mixed_list: list) -> float:
    """Score mixed-emotion items: 0.5 for one match, 1.0 for two+."""
    model_lower = model_emotion.lower().strip()
    matches = sum(1 for e in mixed_list if e.lower() in model_lower or
                  any(s in model_lower for s in EMOTION_SYNONYMS.get(e.lower(), [])))
    if matches >= 2:
        return 1.0
    elif matches >= 1:
        return 0.5
    return 0.0


def score_masked_emotion(model_emotion: str, surface: str, real: str) -> float:
    """Score masked-emotion items: 1.0 for real, 0.0 for surface, 0.5 for acknowledging mask."""
    model_lower = model_emotion.lower().strip()
    # Check if model identified the real emotion
    if emotion_match(model_emotion, real):
        return 1.0
    # Check if model mentions masking/performing/hiding
    mask_words = ["mask", "hiding", "performing", "pretending", "fake", "forced", "facade"]
    if any(w in model_lower for w in mask_words):
        return 0.5
    # If model just says the surface emotion, that's wrong
    if emotion_match(model_emotion, surface):
        return 0.0
    return 0.25  # unrecognized but not the surface trap


@kbench.task(name="Emotional Prosody in Text")
def social_cog_emotional_prosody(llm) -> float:
    """
    Emotional Prosody in Text Benchmark.
    
    Detect emotional tone shifts in written dialogue.
    
    Score = 0.40 * shift_detection + 0.30 * emotion_labeling
          + 0.20 * trigger_id + 0.10 * (1 - false_alarm)
    """
    results = []
    
    for item in PROSODY_ITEMS:
        with kbench.chats.new(f"prosody_{item['id']}"):
            # Format dialogue
            dialogue_text = "\n".join(
                f"Turn {i+1} [{d['speaker']}]: {d['text']}"
                for i, d in enumerate(item["dialogue"])
            )
            
            prompt = (
                f"Read this dialogue carefully and analyze the emotional tone.\n\n"
                f"Dialogue:\n{dialogue_text}\n\n"
                f"Questions:\n"
                f"1. Is there a significant emotional tone SHIFT in this dialogue? (yes/no)\n"
                f"2. If yes, at which turn number does the shift occur?\n"
                f"3. What was the emotional tone BEFORE the shift?\n"
                f"4. What was the emotional tone AFTER the shift?\n"
                f"5. What triggered the shift?\n\n"
                f"Respond with ONLY a JSON object:\n"
                f'{{\n'
                f'  "has_shift": true/false,\n'
                f'  "shift_turn": <turn number or 0>,\n'
                f'  "emotion_before": "<emotion>",\n'
                f'  "emotion_after": "<emotion>",\n'
                f'  "trigger": "<what caused the shift>",\n'
                f'  "confidence": <0-100>\n'
                f'}}'
            )
            
            for _retry in range(3):
                try:
                    response = llm.prompt(prompt, schema=ProsodyResponse)
                    model_has_shift = response.has_shift
                    model_shift_turn = response.shift_turn
                    model_before = response.emotion_before
                    model_after = response.emotion_after
                    model_trigger = response.trigger
                    break
                except Exception:
                    if _retry == 2:
                        # Last resort fallback
                        raw = llm.prompt(prompt)
                        try:
                            cleaned = _strip_think(raw)
                            cleaned = re.sub(r'//.*', '', cleaned)
                            parsed = json.loads(re.search(r'\{.*\}', cleaned, re.DOTALL).group())
                            model_has_shift = bool(parsed.get("has_shift", True))
                            model_shift_turn = int(parsed.get("shift_turn", 0))
                            model_before = str(parsed.get("emotion_before", ""))
                            model_after = str(parsed.get("emotion_after", ""))
                            model_trigger = str(parsed.get("trigger", ""))
                        except Exception:
                            model_has_shift = True
                            model_shift_turn = 0
                            model_before = ""
                            model_after = ""
                            model_trigger = ""
            
            result = {
                "id": item["id"],
                "has_shift_actual": item["has_shift"],
                "has_shift_model": model_has_shift,
            }
            
            if item["has_shift"]:
                # Check shift detection
                result["shift_detected"] = model_has_shift
                
                # Check turn identification — exact match required for all items
                actual_turn = item["shift_turn"]
                result["turn_correct"] = model_shift_turn == actual_turn
                
                # Check emotion labels — special handling for mixed/masked emotions
                # Strict emotion matching for hard items
                difficulty = item.get("difficulty", "standard")
                use_strict = difficulty in ("expert", "camouflaged", "very_hard", "extreme")
                result["before_correct"] = emotion_match(model_before, item["emotion_before"], strict=use_strict)
                
                if "mixed_emotions" in item:
                    result["after_score"] = score_mixed_emotions(model_after, item["mixed_emotions"])
                    result["after_correct"] = result["after_score"] >= 0.5
                elif "surface_emotion" in item and "real_emotion" in item:
                    result["after_score"] = score_masked_emotion(model_after, item["surface_emotion"], item["real_emotion"])
                    result["after_correct"] = result["after_score"] >= 0.5
                else:
                    result["after_correct"] = emotion_match(model_after, item["emotion_after"], strict=use_strict)
                    result["after_score"] = 1.0 if result["after_correct"] else 0.0
                
                # Check trigger: flexible semantic scoring
                # Score 1.0 if core concept words match, partial for partial
                stop_words = {"a", "an", "the", "of", "in", "to", "and", "or", "is", "was", "that", "for", "on", "with", "as", "at", "by", "from", "are", "be", "been", "were", "this", "it", "its"}
                trigger_words = set(item["trigger"].lower().split()) - stop_words
                model_trigger_words = set(model_trigger.lower().split()) - stop_words
                overlap = len(trigger_words & model_trigger_words)
                # Any single content word match gives partial credit
                result["trigger_score"] = min(1.0, overlap / max(1, len(trigger_words) * 0.4))
            else:
                # Control: should detect NO shift
                result["correct_no_shift"] = not model_has_shift
            
            results.append(result)
    
    # ─── Compute Metrics ─────
    shift_items = [r for r in results if r["has_shift_actual"]]
    control_items = [r for r in results if not r["has_shift_actual"]]
    item_lookup = {item["id"]: item for item in PROSODY_ITEMS}
    
    # Categorize by difficulty tier
    easy_items = [r for r in shift_items if item_lookup.get(r["id"], {}).get("difficulty") == "easy"]
    medium_items = [r for r in shift_items if item_lookup.get(r["id"], {}).get("difficulty", "standard") in ("standard", "subtle", None)]
    hard_items = [r for r in shift_items if item_lookup.get(r["id"], {}).get("difficulty") in ("expert", "camouflaged", "very_hard", "extreme")]
    
    def compute_shift_metrics(items, strict_turn=False):
        if not items:
            return 0, 0, 0, 0
        detection = sum(1 for r in items if r.get("shift_detected", False)) / len(items)
        e_scores = []
        t_scores = []
        turn_scores = []
        for r in items:
            if r.get("shift_detected", False):
                after_s = r.get("after_score", 1.0 if r.get("after_correct", False) else 0.0)
                before_s = 1.0 if r.get("before_correct", False) else 0.0
                e_scores.append((before_s + after_s) / 2)
                t_scores.append(r.get("trigger_score", 0))
                turn_scores.append(1.0 if r.get("turn_correct", False) else 0.0)
        emotion = np.mean(e_scores) if e_scores else 0
        trigger = np.mean(t_scores) if t_scores else 0
        turn_acc = np.mean(turn_scores) if turn_scores else 0
        return detection, emotion, trigger, turn_acc
    
    easy_det, easy_emo, _, easy_turn = compute_shift_metrics(easy_items)
    med_det, med_emo, med_trig, med_turn = compute_shift_metrics(medium_items)
    hard_det, hard_emo, hard_trig, hard_turn = compute_shift_metrics(hard_items, strict_turn=True)
    
    # False alarm rate
    adv_controls = [r for r in control_items if item_lookup.get(r["id"], {}).get("difficulty") == "adversarial_control"]
    plain_controls = [r for r in control_items if item_lookup.get(r["id"], {}).get("difficulty") != "adversarial_control"]
    plain_fa = sum(1 for r in plain_controls if not r.get("correct_no_shift", True)) / len(plain_controls) if plain_controls else 0
    adv_fa = sum(1 for r in adv_controls if not r.get("correct_no_shift", True)) / len(adv_controls) if adv_controls else 0
    false_alarm_rate = 0.4 * plain_fa + 0.6 * adv_fa
    
    # Three-tier composite — redesigned for maximum discrimination
    # Key insight: trigger identification has widest spread across models; weight heavily
    # Easy: mostly emotion (baseline check)
    easy_score = float(easy_emo)
    # Medium: trigger-heavy (trigger shows 3-4x variation between strong/weak models)
    medium_score = 0.30 * float(med_emo) + 0.70 * float(med_trig)
    # Hard tier: trigger + emotion combined, both required
    hard_perfect_count = 0
    hard_trigger_total = 0.0
    hard_trigger_high = 0  # items with trigger_score >= 0.30
    for r in hard_items:
        if r.get("shift_detected", False):
            after_s = r.get("after_score", 1.0 if r.get("after_correct", False) else 0.0)
            before_s = 1.0 if r.get("before_correct", False) else 0.0
            perfect = before_s * after_s  # both must be right
            hard_perfect_count += perfect
            trig_s = r.get("trigger_score", 0)
            hard_trigger_total += trig_s
            if trig_s >= 0.30:
                hard_trigger_high += 1
    hard_emo_strict = hard_perfect_count / len(hard_items) if hard_items else 0
    hard_trig_mean = hard_trigger_total / len(hard_items) if hard_items else 0
    hard_trig_high_frac = hard_trigger_high / len(hard_items) if hard_items else 0
    # Hard score: 40% on trigger mean + 30% on high-trigger fraction + 30% on emotion
    hard_score = max(0.0, 0.40 * float(hard_trig_mean) + 0.30 * float(hard_trig_high_frac) + 0.30 * float(hard_emo_strict))
    
    # Final composite: sqrt of hard tier amplifies discrimination
    # Empirically produces std >= 0.08 across 10 models
    score = round(hard_score**0.5 * 0.65 + medium_score * 0.35, 4)
    
    # ─── Logging ─────
    print(f"\n{'='*60}")
    print(f"EMOTIONAL PROSODY IN TEXT RESULTS")
    print(f"{'='*60}")
    
    for r in results:
        if r["has_shift_actual"]:
            detected = "✓" if r.get("shift_detected") else "✗"
            emotions = f"before={'✓' if r.get('before_correct') else '✗'} after={'✓' if r.get('after_correct') else '✗'}"
            trigger = f"trigger={r.get('trigger_score', 0):.2f}"
            print(f"  {detected} {r['id']}: {emotions} {trigger}")
        else:
            correct = "✓ NO SHIFT" if r.get("correct_no_shift") else "✗ FALSE ALARM"
            print(f"  {correct} {r['id']}")
    
    print(f"\n--- Summary ---")
    print(f"Easy detection:     {easy_det:.2%} (N={len(easy_items)})")
    print(f"Easy emotion:       {easy_emo:.2%}")
    print(f"Medium detection:   {med_det:.2%} (N={len(medium_items)})")
    print(f"Medium emotion:     {med_emo:.2%}")
    print(f"Medium trigger:     {med_trig:.2%}")
    print(f"Medium turn acc:    {med_turn:.2%}")
    print(f"Hard detection:     {hard_det:.2%} (N={len(hard_items)})")
    print(f"Hard emotion:       {hard_emo:.2%}")
    print(f"Hard trigger:       {hard_trig:.2%}")
    print(f"Hard turn accuracy: {hard_turn:.2%}")
    print(f"False alarm rate:   {false_alarm_rate:.2%}")
    print(f"Easy tier score:    {easy_score:.4f} (weight 0.10)")
    print(f"Medium tier score:  {medium_score:.4f} (weight 0.30)")
    print(f"Hard tier score:    {hard_score:.4f} (weight 0.60)")
    print(f"Composite score:    {score:.4f}")
    
    return score




In [ ]:
social_cog_emotional_prosody.run(llm=kbench.llm)


In [ ]:
%choose Emotional Prosody in Text